In [34]:
import pandas as pd
from collections import deque

In [35]:
LEFT = 0
ISLAND = 1
RIGHT = 2

RESEARCHER = 0
AGENT = 1

In [36]:
# hard requirement: a reseacher can't share the same position with a different agent
def is_valid(pairs):
    for location in [LEFT, ISLAND, RIGHT]:
        agents_at_loc = {i for i, (r, a) in enumerate(pairs) if a == location}
        researchers_at_loc = {i for i, (r, a) in enumerate(pairs) if r == location}

        if agents_at_loc:
            for r_idx in researchers_at_loc:
                if r_idx not in agents_at_loc:
                    return False
                
    return True

In [ ]:
def solve(n: int):
    # pairs = n tuples of (researcher position, agent position)
    initial_pairs = tuple(sorted([(LEFT, LEFT) for _ in range(n)]))
    goal_pairs = tuple(sorted([(RIGHT, RIGHT) for _ in range(n)]))

    # state = (boat position, pairs)
    initial_state = (LEFT, initial_pairs)
    goal_state = (RIGHT, goal_pairs)

    # (state, distance)
    queue = deque([(initial_state, 0)])
    visited = {initial_state}

    while queue:
        # 0. get current state info
        (boat_loc, pairs), dist = queue.popleft()

        # 1. did everyone end up on the RIGHT side?
        if(boat_loc, pairs) == goal_state:
            return dist
        
        # 2. generate all possible transitions

        # all possible next locations
        next_boat_locs = [ISLAND] if boat_loc != ISLAND else [LEFT, RIGHT]

        # find all people at the current boat location
        # these are the only ones who can go on the boat
        eligible_people = []
        for i, (researcher_pos, agent_pos) in enumerate(pairs):
            if researcher_pos == boat_loc:
                eligible_people.append((i, RESEARCHER))
            if agent_pos == boat_loc:
                eligible_people.append((i, AGENT))

        # try to move i-th and j-th people
        for i in range(len(eligible_people)):
            for j in range(i, len(eligible_people)):
                # j starts from i, so first time we transport only 1 person
                passengers = []
                if j == i:
                    passengers = [eligible_people[i]]
                else:
                    passengers = [eligible_people[i], eligible_people[j]]

                for next_loc in next_boat_locs:
                    # try each possible next location

                    # transitions = all eligible people (select 1 or 2) and all eligibile locations

                    # 3. update the pairs for the current transition
                    new_pairs = list(pairs) # conver to list so we can mutate
                    for p_idx, p_type in passengers:
                        # for passenger p, find where its companion is
                        researcher_pos, agent_pos = pairs[p_idx]

                        # change the position of the one being moved, the other companion stays the same
                        if p_type == RESEARCHER:
                            new_pairs[p_idx] = (next_loc, agent_pos)
                        else:
                            new_pairs[p_idx] = (researcher_pos, next_loc)

                    # 4. add the new generated state to the queue
                    new_pairs = tuple(sorted(new_pairs))
                    if is_valid(new_pairs):
                        new_state = (next_loc, new_pairs)
                        if new_state not in visited:
                            visited.add(new_state)
                            queue.append((new_state, dist+1))

    return -1

In [51]:
solve(3)

18

In [52]:
subtask1 = [solve(3)]
subtask2 = [solve(n) for n in range(4, 9)]

In [53]:
submission = pd.DataFrame({
    "subtaskID": [1, 2, 2, 2, 2, 2],
    "datapointID": list(range(3, 9)),
    "answer": subtask1 + subtask2
})

submission.head()

,subtaskID,datapointID,answer
0,1,3,18
1,2,4,26
2,2,5,34
3,2,6,42
4,2,7,50


In [54]:
submission.to_csv("submission.csv", index=False)